In [1]:

import json
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import datasets, transforms, models
from PIL import Image

import numpy as np
import plotly.graph_objects as go
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

import optuna
from optuna.pruners import MedianPruner
import pickle

In [2]:
user_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [4]:
DATA_DIR = Path('data')
TRAIN_DIR = DATA_DIR / 'Train'
TEST_DIR = DATA_DIR / 'Test'
VALI_DIR = DATA_DIR / 'Validation'

In [5]:
IMG_SIZE = 224
BATCH_SIZE = 32
LEARN_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 20
PATIENCE = 5
VAL_SPLIT = 0.1

In [6]:
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents = True, exist_ok = True)

assert TRAIN_DIR.exists() and VALI_DIR.exists() and TEST_DIR.exists(), 'Train/Validation/Test folders not found.'
print('Using data at:', DATA_DIR.resolve())

Using data at: D:\Coding\Uni Marburg\gender-recogniser\data


In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale = (0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness = 0.1, contrast = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

eval_tf = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

ds_train = datasets.ImageFolder(TRAIN_DIR, transform = train_tf)
class_names = ds_train.classes
num_classes = len(class_names)

y_full = np.array([y for _, y in ds_train.samples])
strat_split = StratifiedShuffleSplit(n_splits = 1, test_size = VAL_SPLIT, random_state = SEED)
train_idx, val_idx = next(strat_split.split(np.zeros(len(y_full)), y_full))

ds_train = Subset(datasets.ImageFolder(TRAIN_DIR, transform = train_tf), train_idx)
ds_val = Subset(datasets.ImageFolder(TRAIN_DIR, transform = eval_tf), val_idx)
ds_test = datasets.ImageFolder(TEST_DIR, transform = eval_tf)


In [8]:
print('Classes:', class_names, ' (num_classes =', num_classes, ')')

print(f'Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')

Classes: ['Female', 'Male']  (num_classes = 2 )
Train: 10021 | Val: 1114 | Test: 1279


In [9]:
class_counts = np.bincount(y_full, minlength = num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1.0)

In [10]:
global sample_weights

In [11]:
train_targets = y_full[train_idx]
sample_weights = [class_weights[y] for y in train_targets]

sampler = WeightedRandomSampler(weights = [float(w) for w in sample_weights],
                                num_samples = len(sample_weights), replacement = True)

dl_train = DataLoader(ds_train, batch_size = BATCH_SIZE, sampler = sampler, num_workers = 2, pin_memory = True)
dl_test = DataLoader(ds_test, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)
dl_val = DataLoader(ds_val, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)

In [12]:
def build_model(num_classes : int, lr : float = LEARN_RATE, wd : float = WEIGHT_DECAY):
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights = weights)
    except Exception:
        model = models.resnet18(pretrained = True)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    model = model.to(user_device)

    cw = torch.tensor(class_weights / class_weights.sum() * num_classes, dtype = torch.float32, device = user_device)
    criterion = nn.CrossEntropyLoss(weight = cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 2)
    return model, criterion, optimizer, scheduler

model, criterion, optimizer, scheduler = build_model(num_classes)

In [13]:
def epoch_step(loader, train : bool):
    model.train() if train else model.eval()

    running_loss, running_corrects, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    if len(loader) == 0:
        return 0.0, 0.0, np.array([], dtype = np.int64), np.array([], dtype = np.int64)

    for xb, yb in loader:
        xb, yb = xb.to(user_device), yb.to(user_device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        preds = torch.argmax(logits, dim = 1).view(-1)
        running_loss += loss.item() * xb.size(0)
 
        running_corrects += (preds == yb).sum().item()
        total += xb.size(0)

        preds_flat = torch.argmax(logits, dim = 1).view(-1) 
        yb_flat = yb.view(-1) 

        if preds_flat.numel() == 0:
            continue

        all_preds.extend(preds_flat.detach().cpu().numpy().ravel().tolist())
        all_targets.extend(yb_flat.detach().cpu().numpy().ravel().tolist())
    
    avg_loss = running_loss / max(total, 1)
    avg_acc = running_corrects / max(total, 1)
    
    return avg_loss, avg_acc, np.array(all_preds, dtype = np.int64), np.array(all_targets, dtype = np.int64)

def train_model(epoch = EPOCHS, patience = PATIENCE):
    best_val, best_state, patience_left = float('inf'), None, patience
    hist = {'train_loss' : [],
            'train_acc' : [],
            'val_loss' : [],
            'val_acc' : []}
    
    for e in range(1, epoch + 1):
        tr_loss, tr_acc, _, _ = epoch_step(dl_train, True)
        val_loss, val_acc, _, _ = epoch_step(dl_val, False)

        scheduler.step(val_loss)
        hist['train_loss'].append(tr_loss)
        hist['val_loss'].append(val_loss)
        hist['train_acc'].append(tr_acc)
        hist['val_acc'].append(val_acc)
        print(f'Epoch {e:02d}/{epoch} | train_loss {tr_loss:.4f} val_loss {val_loss:.4f} | train_acc {tr_acc:.3f} val_acc {val_acc:.3f}')
        if val_loss < best_val - 1e-4:
            best_val, best_state, patience_left = val_loss, {k: v.cpu() for k, v in model.state_dict().items()}, patience
            torch.save(best_state, OUT_DIR/'normal_model.pt')
            with open(OUT_DIR/'labels.json', 'w') as f:
                json.dump({i: n for i, n in enumerate(class_names)}, f)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('--Early stopping--')
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(user_device)
    
    return hist

In [14]:
if len(dl_val) == 0:
    print('Warning: validation split is empty')
    PATIENCE = 0

history = train_model()

d:\Coding\Uni Marburg\gender-recogniser\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/20 | train_loss 0.2077 val_loss 0.1690 | train_acc 0.915 val_acc 0.938
Epoch 02/20 | train_loss 0.1352 val_loss 0.2727 | train_acc 0.952 val_acc 0.903
Epoch 03/20 | train_loss 0.1325 val_loss 0.1444 | train_acc 0.954 val_acc 0.959
Epoch 04/20 | train_loss 0.0974 val_loss 0.1622 | train_acc 0.967 val_acc 0.952
Epoch 05/20 | train_loss 0.0951 val_loss 0.1837 | train_acc 0.967 val_acc 0.940
Epoch 06/20 | train_loss 0.0908 val_loss 0.1854 | train_acc 0.969 val_acc 0.946
Epoch 07/20 | train_loss 0.0609 val_loss 0.1844 | train_acc 0.979 val_acc 0.954
Epoch 08/20 | train_loss 0.0519 val_loss 0.1828 | train_acc 0.981 val_acc 0.952
--Early stopping--


In [15]:
best_model_path = OUT_DIR / 'normal_model.pt'
if best_model_path.exists():
    model.load_state_dict(torch.load(best_model_path, map_location = user_device))
    model.to(user_device)
    model.eval()
    print(f"Loaded best model from {best_model_path}")
else:
    print("Best model file not found.")

Loaded best model from outputs\normal_model.pt


In [16]:
def subset_targets(subset) -> np.ndarray:
    if isinstance(subset, Subset):
        samples = subset.dataset.samples
        idxs = subset.indices
        return np.array([samples[i][1] for i in idxs], dtype = np.int64)
    else:
        return np.array([y for _, y in subset.samples], dtype = np.int64)

def create_batch_loader(batch_size:int):
    train_targets_local = subset_targets(ds_train)
    class_counts_local = np.bincount(train_targets_local, minlength = num_classes)
    class_weights_local = 1.0 / np.maximum(class_counts_local, 1)
    sample_weights_local = [float(class_weights_local[y]) for y in train_targets_local]
    sampler_local = WeightedRandomSampler(sample_weights_local, num_samples = len(sample_weights_local), replacement = True)
    dl_train_local = DataLoader(ds_train, batch_size = batch_size, sampler = sampler_local, num_workers = 2, pin_memory = True)
    dl_val_local = DataLoader(ds_val, batch_size = batch_size, shuffle = False,  num_workers = 2, pin_memory = True)
    return dl_train_local, dl_val_local, class_weights_local

def model_builder(num_classes:int, lr:float, wd:float):
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model_t = models.resnet18(weights = weights)
    except Exception:
        model_t = models.resnet18(pretrained = True)
    in_features = model_t.fc.in_features
    model_t.fc = nn.Linear(in_features, num_classes)
    model_t = model_t.to(user_device)

    cw_t = torch.tensor(class_weights / class_weights.sum() * num_classes, dtype = torch.float32, device = user_device)
    criterion_t = nn.CrossEntropyLoss(weight = cw_t)
    optimizer_t = torch.optim.AdamW(model_t.parameters(), lr = lr, weight_decay = wd)
    scheduler_t = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_t, mode = 'min', factor = 0.5, patience = 2)
    
    return model_t, criterion_t, optimizer_t, scheduler_t

In [17]:
def objective(trial: optuna.trial.Trial) -> float:
    lr = trial.suggest_float('lr', 1e-5, 3e-3, log = True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log = True)
    bs = trial.suggest_categorical('batch_size', [16, 32, 64])
    ep = trial.suggest_int('epochs', 6, 18)

    dl_train_t, dl_val_t, class_weights_t = create_batch_loader(bs)
    model_t, criterion_t, optimizer_t, scheduler_t = model_builder(num_classes, lr, wd)

    best_val_acc = 0.0
    best_state = None
    patience = 4

    for epoch in range(1, ep + 1):
        tr_loss, tr_acc, _, _   = epoch_step(dl_train_t, train = True)
        val_loss, val_acc, _, _ = epoch_step(dl_val_t, train = False)

        scheduler_t.step(val_loss)
        trial.report(val_acc, step = epoch)

        if val_acc > best_val_acc + 1e-6:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu() for k, v in model_t.state_dict().items()}
            patience = 4
        else:
            patience -= 1
            if patience <= 0:
                break

        if trial.should_prune():
            raise optuna.TrialPruned()

    trial_ckpt = OUT_DIR / f"trial_{trial.number}_best.pt"
    if best_state is not None:
        torch.save(best_state, trial_ckpt)

    return float(best_val_acc)

In [18]:
study = optuna.create_study(direction = "maximize", pruner = MedianPruner(n_startup_trials = 3, n_warmup_steps = 2))
study.optimize(objective, n_trials = 15, show_progress_bar = False)

print("Best trial:", study.best_trial.number)
print("Best value (val_acc):", study.best_value)
print("Best params:", study.best_params)

[I 2025-08-13 15:35:26,338] A new study created in memory with name: no-name-196e6bac-3c3b-490d-8b90-520911cfb6a0
[I 2025-08-13 18:00:56,023] Trial 0 finished with value: 0.9614003590664273 and parameters: {'lr': 0.002478229418217634, 'weight_decay': 1.4920154696770221e-05, 'batch_size': 16, 'epochs': 13}. Best is trial 0 with value: 0.9614003590664273.
[W 2025-08-13 19:19:44,658] Trial 1 failed with parameters: {'lr': 0.0025644936802368275, 'weight_decay': 4.103744608058234e-05, 'batch_size': 64, 'epochs': 8} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\Coding\Uni Marburg\gender-recogniser\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\sshre\AppData\Local\Temp\ipykernel_8228\3705330011.py", line 15, in objective
    tr_loss, tr_acc, _, _   = epoch_step(dl_train_t, train = True)
                              ^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
best = study.best_params
dl_train_opt, dl_val_opt, class_weights_opt = create_batch_loader(best["batch_size"])
model_opt, criterion_opt, optimizer_opt, scheduler_opt = model_builder(num_classes, best["lr"], best["weight_decay"])

In [ ]:
def train_model_with_objects(model, criterion, optimizer, scheduler, epochs=EPOCHS, patience=PATIENCE):
    best_val = float('inf')
    best_state = None
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    patience_left = patience
    for e in range(1, epochs + 1):
        tr_loss, tr_acc, _, _ = epoch_step(dl_train_opt, True)
        val_loss, val_acc, _, _ = epoch_step(dl_val_opt, False)
        scheduler.step(val_loss)
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(val_loss)
        hist['train_acc'].append(tr_acc); hist['val_acc'].append(val_acc)
        print(f'[OPT] Epoch {e:02d} | tr_loss {tr_loss:.4f} val_loss {val_loss:.4f} | tr_acc {tr_acc:.3f} val_acc {val_acc:.3f}')
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('-- [OPT] Early stopping --')
                break
    if best_state is not None:
        model.load_state_dict(best_state) 
        model.to(user_device)
    return hist, model


In [ ]:
hist_opt, model_opt = train_model_with_objects(
    model_opt, criterion_opt, optimizer_opt, scheduler_opt,
    epochs = min(max(best["epochs"] + 2, best["epochs"]), 25),
    patience = 5
)

In [ ]:
torch.save({k: v.cpu() for k, v in model_opt.state_dict().items()}, OUT_DIR / "optuna_model.pt")

with open(OUT_DIR / "history.pkl", "wb") as f:
    pickle.dump(hist_opt, f)

In [ ]:
def result_report(t_loss, t_acc, t_preds, t_targets, type_str):
    print(f'Test — loss: {t_loss:.4f} | acc: {t_acc:.3f}')
    print(f'\nClassification Report {type_str}:')
    print(classification_report(t_targets, t_preds, target_names = class_names))

    cm = confusion_matrix(t_targets, t_preds, labels = list(range(num_classes)))
    fig = go.Figure(data = go.Heatmap(
        z = cm, x = class_names, y = class_names, colorscale = 'Viridis',
        hovertemplate = 'Predicted: %{x}<br>Actual: %{y}<br>Count: %{z}<extra></extra>'))
    fig.update_layout(title = 'Confusion Matrix (Test)', xaxis_title = 'Predicted', yaxis_title = 'Actual')
    fig.show()

In [ ]:
test_loss, test_acc, test_preds, test_targets = epoch_step(dl_test, False)

result_report(test_loss, test_acc, test_preds, test_targets, "pre-optuna")

In [ ]:
test_loss_opt, test_acc_opt, test_preds_opt, test_targets_opt = epoch_step(dl_test, False)

result_report(test_loss_opt, test_acc_opt, test_preds_opt, test_targets_opt, "post-optuna")

In [ ]:
def gather_images(root: Path):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
    return [p for p in root.rglob('*') if p.suffix.lower() in exts]

def predict_image(path: Path):
    model.eval()
    with torch.no_grad():
        img = Image.open(path).convert('RGB')
        x = transforms.Compose([
            transforms.Resize((256,256)),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])(img)
        x = x.unsqueeze(0).to(user_device)
        probs = torch.softmax(model(x), dim = 1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return class_names[pred_idx], probs

img_paths = gather_images(VALI_DIR)
print('Validation images (unlabeled):', len(img_paths))

In [ ]:
rows = []
for p in img_paths:
    label, probs = predict_image(p)
    row = {'filepath': str(p), 'pred_class': label}
    for i, cn in enumerate(class_names):
        row[f'prob_{cn}'] = f"{float(probs[i]):.4f}"
    rows.append(row)

In [ ]:
df = pd.DataFrame(rows).sort_values('filepath')
csv_path = OUT_DIR / 'validation_predictions.csv'
df.to_csv(csv_path, index = False)
print('Wrote', csv_path.resolve())
df.head()

In [ ]:
x = list(range(1, len(history['train_loss']) + 1))

In [ ]:
fig1 = go.Figure()
fig1.add_scatter(x = x, y = history['train_loss'], mode = 'lines+markers', name = 'train_loss')
fig1.add_scatter(x = x, y = history['val_loss'], mode = 'lines+markers', name = 'val_loss')
fig1.update_layout(title = 'Loss', xaxis_title = 'Epoch', yaxis_title = 'Loss')
fig1.show()

In [ ]:
fig2 = go.Figure()
fig2.add_scatter(x = x, y = history['train_acc'], mode = 'lines+markers', name = 'train_acc')
fig2.add_scatter(x = x, y = history['val_acc'], mode = 'lines+markers', name = 'val_acc')
fig2.update_layout(title = 'Accuracy', xaxis_title = 'Epoch', yaxis_title = 'Accuracy')
fig2.show()